~/.claude/CLAUDE.md: Personal preferences that apply to all projects. Things like preferred code style, personal shortcuts, or default behaviors you always want.
./CLAUDE.md: Project-wide conventions shared by the entire team.
subdir/CLAUDE.md: Subdirectory-specific rules that apply only when working within that directory tree.

# The @import Pattern
For large projects, a single CLAUDE.md file becomes unwieldy. The @import directive lets you split configuration into modular files.

# Project conventions
Use TypeScript strict mode for all new files.
Run `npm run lint` before committing.
@import .claude/rules/testing.md
@import .claude/rules/security.md
@import .claude/rules/api-conventions.md

# Path-Scoped Rules: .claude/rules/
The .claude/rules/ directory supports YAML frontmatter with a paths field that scopes the rule to specific file patterns. This is a more surgical alternative to directory-level CLAUDE.md files.

---
paths: ["terraform/**/*", "*.tf"]
---

# Terraform Conventions
- Always use `terraform fmt` before committing
- Module sources must use version pins
- State backend must be S3, never local
- Tag all resources with team and environment

# Commands, Skills & Path Rules
**Custom Commands**: Custom commands extend Claude Code with project-specific or personal workflows. They live as Markdown files in a .claude/commands/ directory.

# .claude/commands/review.md (example)
Review the changes in the current branch against main.
Focus on:
1. Type safety violations
2. Missing error handling
3. Security concerns (SQL injection, XSS)
4. Test coverage for new code paths

Use `git diff main...HEAD` to see the changes.
Provide a structured review with severity levels.

# Skills: SKILL.md Frontmatter
Skills are enhanced commands with metadata that controls execution context. The SKILL.md frontmatter defines critical behavior

# SKILL.md Frontmatter
---
name: deploy-staging
context: fork
allowed-tools: [Bash, Read, Glob]
argument-hint: "branch name to deploy"
---

# Deploy to Staging
Deploy the specified branch to the staging environment.
1. Verify the branch exists
2. Run the test suite
3. Build the Docker image
4. Push to staging registry
5. Trigger deployment via kubectl


# Key frontmatter fields for the exam:
1. context: fork -- Runs the skill in a separate context window (sub-agent), so it does not pollute the main conversation. Use for heavy research or long-running tasks.
2. allowed-tools -- Restricts which tools the skill can use. Critical for security-sensitive skills that should not have shell access.
3. argument-hint -- Tells the user what argument to pass when invoking the skill.

# 4. Plan Mode vs Direct Execution FROM HERE
# The explore sub-agent
// Step 1: Plan mode -- understand the architecture
"Plan how to migrate the authentication system from
 sessions to JWT tokens."

// Step 2: Explore sub-agent -- research details
"Use the Agent tool to find all session-related code
 across the codebase and map the dependencies."

// Step 3: Direct execution -- implement
"Now implement the JWT middleware as planned."

# The -p Flag: Non-Interactive Mode
# Non-interactive code review
claude -p "Review the changes in this PR for security issues. \
  Focus on SQL injection, XSS, and auth bypasses."

# With structured output
claude -p "Analyze this codebase for type safety issues" \
  --output-format json \
  --json-schema '{"type":"object","properties":{"issues":{"type":"array"}}}'

# Structured Output: --output-format json
{
  "type": "object",
  "properties": {
    "summary": { "type": "string" },
    "issues": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "severity": { "type": "string",
            "enum": ["critical", "high", "medium", "low"] },
          "file": { "type": "string" },
          "line": { "type": "integer" },
          "description": { "type": "string" },
          "suggestion": { "type": "string" }
        }
      }
    },
    "approved": { "type": "boolean" }
  }
}

# Including Prior Review Findings
# First review
claude -p "Review this PR" --output-format json > review-1.json

# After fixes, include prior findings
claude -p "Review this PR. These issues were already found
  and fixed in a previous review: $(cat review-1.json).
  Do NOT report these again. Focus on new issues only." \
  --output-format json > review-2.json

# Test-Driven Iteration
The interview pattern uses Claude in a loop: write code, run tests, feed failures back, iterate until all tests pass. This is especially powerful in CI.
1. Define concrete input/output examples
2. Generate implementation
3. Run tests, feed back failures
4. Repeat until green



##### Configure and execute workflow pipeline
1. Create the workflow file in VS Code: .github/workflows/ci-cd.yml
2. Add your API key to GitHub
3. Push from VS Code
4. Monitor the run

In [ ]:
# Interactive: CI/CD Pipeline Builder
name: CI/CD Pipeline

on:
  pull_request:
    branches: [main]
  push:
    branches: [main]

jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-node@v4
        with:
          node-version: 20
      - run: npm ci
      - run: npm run lint

  test:
    runs-on: ubuntu-latest
    needs: lint
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-node@v4
        with:
          node-version: 20
      - run: npm ci
      - run: npm test -- --coverage

  claude-review:
    runs-on: ubuntu-latest
    needs: test
    if: github.event_name == 'pull_request'
    timeout-minutes: 5
    steps:
      - uses: actions/checkout@v4
        with:
          fetch-depth: 0
      - name: Run Claude Code Review
        env:
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
        run: |
          claude -p "Review the changes in this PR. Focus on: 1. Security vulnerabilities 2. Type safety issues 3. Missing error handling 4. Performance concerns  Output a structured JSON review." \
            --model claude-sonnet-4-20250514 \
            --output-format json \
            --allowedTools "Read,Grep"

  deploy:
    runs-on: ubuntu-latest
    needs: [test, claude-review]
    if: github.ref == 'refs/heads/main'
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-node@v4
        with:
          node-version: 20
      - run: npm ci
      - run: npm run deploy:staging